# Reviewer-robust full analysis notebook

This notebook is intended to be the **single rerunnable revision analysis** for the CCM
review response. It reproduces the paper-facing results across all four datasets and both
outcomes, then adds the robustness checks requested by reviewers:

- S-learners: XGBoost, L1-regularized logistic S-learner, neural-network S-learner, and BART.
- R-learner/forest: `CausalForestDML` with cross-fitted nuisance models.
- Stratified K-fold cross-validation, stratified jointly by treatment and outcome, with
  out-of-fold predictions for every patient.
- All learned preprocessing fit on each fold's training data only and applied unchanged to
  that fold's validation data.
- LR CATE-by-TTM interaction tests, CATE interval coverage, GATES, calibration, AUROC/Brier, and variable importance.
- Observational-cohort IPW versions of GATES and the CATE-by-TTM interaction diagnostic.
- Output CSVs/figures that can be pasted into `RESPONSE_TO_REVIEWERS.md`.

Run from `pooled/`:

```bash
jupyter nbconvert --to notebook --execute reviewerRobustAnalysis.ipynb \
  --output reviewerRobustAnalysis.executed.ipynb --ExecutePreprocessor.timeout=-1
```

The predictor CSVs are not committed to this repo. This notebook resolves them from the
candidate paths/patterns in `CSV_CANDIDATES` below and prints the selected files at runtime.


In [22]:
# CONFIG
import os
import glob

def env_flag(name, default=True):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() not in {'0', 'false', 'no', 'off'}


SEED = 42
N_SPLITS = int(os.environ.get('N_SPLITS', '5'))
N_BOOT = 500
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95)
CORR_THRESHOLD = 0.70

# Runtime switches. Override on the cluster, e.g. RUN_BART_SLEARNER=0 for a smoke run.
RUN_XGB_SLEARNER = env_flag('RUN_XGB_SLEARNER', True)
RUN_LASSO_SLEARNER = env_flag('RUN_LASSO_SLEARNER', True)
RUN_NEURAL_SLEARNER = env_flag('RUN_NEURAL_SLEARNER', True)
RUN_BART_SLEARNER = env_flag('RUN_BART_SLEARNER', True)
RUN_CAUSAL_FOREST = env_flag('RUN_CAUSAL_FOREST', True)
USE_TF_GPU = env_flag('USE_TF_GPU', False)
NEURAL_BACKEND = os.environ.get('NEURAL_BACKEND', 'keras' if USE_TF_GPU else 'sklearn').lower()

# BART settings mirror the submitted notebooks. Increase draws for final archival runs if time allows.
BART_DRAW_SCALE = float(os.environ.get('BART_DRAW_SCALE', '1.0'))
BART_DRAWS = {
    'eICU': int(50 * BART_DRAW_SCALE),
    'PMAP': int(50 * BART_DRAW_SCALE),
    'MIMIC-IV': int(50 * BART_DRAW_SCALE),
    'HYPERION': int(100 * BART_DRAW_SCALE),
}
BART_TUNE = int(float(os.environ.get('BART_TUNE_SCALE', '1.0')) * 50)
BART_CHAINS = 2
BART_CORES = 1

REPO_ROOT = os.path.abspath('..')
OUTDIR = os.path.abspath('reviewer_robust_outputs')
os.makedirs(OUTDIR, exist_ok=True)

CSV_CANDIDATES = {
    'eICU': [
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictorsDiag.csv'),
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictors*.csv'),
    ],
    'PMAP': [
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_Predictors2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_*2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_Predictors4.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP*Predictors*.csv'),
    ],
    'MIMIC-IV': [
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_*Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_Predictors1.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC*Predictors*.csv'),
    ],
    'HYPERION': [
        os.path.join(REPO_ROOT, 'hyperion', 'predictorsDf.csv'),
        os.path.join(REPO_ROOT, 'hyperion', '*predictors*.csv'),
    ],
}

def resolve_csv(name):
    matches = []
    for candidate in CSV_CANDIDATES[name]:
        hits = sorted(glob.glob(candidate))
        if hits:
            matches.extend(hits)
    if not matches:
        return CSV_CANDIDATES[name][0]
    # Preserve candidate priority while de-duplicating.
    seen = set()
    ordered = []
    for p in matches:
        if p not in seen:
            ordered.append(p)
            seen.add(p)
    return ordered[0]

CSV = {name: resolve_csv(name) for name in CSV_CANDIDATES}
print('Resolved predictor CSVs:')
for name, path in CSV.items():
    print(f'  {name}: {path}')

# TensorFlow on the cluster can fail if it sees a GPU but CUDA compiler tools
# (ptxas/nvlink) are absent. Default neural nets to CPU; opt in with USE_TF_GPU=1.
if not USE_TF_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
    os.environ['XLA_FLAGS'] = ''

DATASETS = [
    {'name': 'eICU',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'PMAP',     'observational': True,  'treatment_col': 'TTM'},
    {'name': 'MIMIC-IV', 'observational': True,  'treatment_col': 'TTM'},
    {'name': 'HYPERION', 'observational': False, 'treatment_col': 'TTM'},
]
OUTCOMES = {
    'mortality': 'mortality',
    'neuro': 'neuro_favorable',
}


Resolved predictor CSVs:
  eICU: /home/mbranda1/ttmhte/eICU/eICUPredictorsDiag.csv
  PMAP: /home/mbranda1/ttmhte/pmap/PMAP_Predictors2.csv
  MIMIC-IV: /home/mbranda1/ttmhte/mimiciv/MIMIC_Predictors.csv
  HYPERION: /home/mbranda1/ttmhte/hyperion/predictorsDf.csv


In [23]:
import os
import ast
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

try:
    from econml.dml import CausalForestDML
except Exception as e:
    CausalForestDML = None
    print('econml unavailable:', e)

if RUN_NEURAL_SLEARNER and NEURAL_BACKEND == 'keras':
    try:
        import tensorflow as tf
        if not USE_TF_GPU:
            try:
                tf.config.set_visible_devices([], 'GPU')
                tf.config.optimizer.set_jit(False)
            except Exception as _tf_gpu_e:
                print('TensorFlow GPU disable warning:', _tf_gpu_e)
        from tensorflow import keras
        from tensorflow.keras import layers
    except Exception as e:
        tf = None
        keras = None
        layers = None
        print('TensorFlow unavailable; neural model will fall back to sklearn MLP:', e)
        NEURAL_BACKEND = 'sklearn'
else:
    tf = None
    keras = None
    layers = None

try:
    import pymc as pm
    import pymc_bart as pmb
except Exception as e:
    pm = None
    pmb = None
    print('PyMC/BART unavailable:', e)

np.random.seed(SEED)
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 80)


## Curated feature sets

These are copied from the current dataset DML/BART notebooks. They keep the analysis anchored
to clinically meaningful, pre-arrest/early-post-arrest variables and avoid training on the full
raw EHR feature space.


In [24]:
CURATED = {
 'eICU': ['gender','age','bmi',
    'nurse_first_Non-Invasive BP Systolic','nurse_first_Non-Invasive BP Diastolic',
    'nurse_first_Non-Invasive BP Mean','nurse_first_Heart Rate','nurse_first_O2 Saturation',
    'lab_first_Respiratory Rate','lab_first_FiO2','nurse_first_GCS Total','nurse_first_Motor',
    'nurse_first_QTc','lab_first_pH','lab_first_paO2','lab_first_paCO2','lab_first_bicarbonate',
    'lab_first_lactate','lab_first_WBC x 1000','lab_first_Hgb','lab_first_platelets x 1000',
    'lab_first_sodium','lab_first_potassium','lab_first_BUN','lab_first_creatinine',
    'lab_first_calcium','lab_first_magnesium','lab_first_glucose','lab_first_troponin - T',
    'diagnosis_initial rhythm: ventricular fibrillation',
    'diagnosis_initial rhythm: ventricular tachycardia',
    'diagnosis_initial rhythm: pulseless electrical activity',
    'diagnosis_initial rhythm: asystole'],
 'PMAP': ['gender','age','first_mGCS','flo_first_r_cpn_glasgow_coma_scale_score',
    'flo_first_bp_systolic','flo_first_bp_diastolic','flo_first_r_map',
    'flo_first_r_ed_pre-arrival_pulse_(heart_rate)','flo_first_r_sao2','flo_first_r_fio2',
    'flo_first_r_sofa_score','flo_first_r_bmi','flo_first_r_pao2','flo_first_r_paco2',
    'flo_first_r_resp_ph','lab_first_lactate','lab_first_troponin','lab_first_hemoglobin',
    'lab_first_platelet_count','lab_first_creatinine,whole_blood','lab_first_glucose,whole_blood',
    'lab_first_potassium,whole_blood','lab_first_sodium,whole_blood','lab_first_calcium,_serum',
    'lab_first_magnesium','lab_first_jhm_ip_qtc_ainterval_(sec)','asystole','pea','VF'],
 'MIMIC-IV': ['gender','age','bmi','first_mGCS',
    'chart_first_heart_rate','chart_first_o2_saturation_pulseoxymetry','chart_first_respiratory_rate',
    'chart_first_fio2_(ch)','chart_first_non_invasive_blood_pressure_systolic',
    'chart_first_non_invasive_blood_pressure_diastolic','chart_first_non_invasive_blood_pressure_mean',
    'chart_first_ph_(arterial)','chart_first_arterial_o2_pressure','chart_first_arterial_co2_pressure',
    'chart_first_lactic_acid','chart_first_wbc','chart_first_hemoglobin','lab_first_platelet_count',
    'chart_first_sodium_(serum)','lab_first_potassium_(serum)','chart_first_bun',
    'chart_first_creatinine_(serum)','chart_first_calcium_non-ionized','chart_first_magnesium',
    'chart_first_glucose_(serum)','lab_first_troponin-t','chart_first_qtc',
    'long_title_ventricular_fibrillation'],
 'HYPERION': ['J0_AGE','J0_SEX','J0_BMI','J0_PAS','J0_PAD','J0_PAM','J0_FC','J0_SPO2',
    'J0_GLASGOW','J0_MOTRICE','J0_RYTHM','J0_NOFLOW','J0_LOWFLOW','J0_IGSII',
    'BIO_LEUCO','BIO_HEMO','BIO_PLAQ','BIO_SODIUM','BIO_POTAS','BIO_UREE','BIO_CREAT',
    'BIO_CALCIUM','BIO_MAGNE','BIO_GLYCEMI','BIO_LACTAT','BIO_TROPO','BIO_PH','BIO_PAO2',
    'BIO_PACO2','BIO_BICARB'],
}

DML_NOTEBOOKS = {
    'eICU': os.path.join(REPO_ROOT, 'eICU', 'eICUAnalysisDML.ipynb'),
    'PMAP': os.path.join(REPO_ROOT, 'pmap', 'PMAPAnalysisDML.ipynb'),
    'MIMIC-IV': os.path.join(REPO_ROOT, 'mimiciv', 'MIMICAnalysisDML.ipynb'),
    'HYPERION': os.path.join(REPO_ROOT, 'hyperion', 'hyperionAnalysisDML.ipynb'),
}


def extract_dml_columns(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        nb = json.load(f)
    lists = []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        if 'columns' not in src:
            continue
        try:
            tree = ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and target.id == 'columns':
                        try:
                            val = ast.literal_eval(node.value)
                        except Exception:
                            continue
                        if isinstance(val, list) and all(isinstance(x, str) for x in val):
                            lists.append(val)
    return max(lists, key=len) if lists else None


PARSED_DML_COLUMN_DATASETS = set()
for _name, _path in DML_NOTEBOOKS.items():
    _cols = extract_dml_columns(_path)
    if _cols:
        CURATED[_name] = _cols
        PARSED_DML_COLUMN_DATASETS.add(_name)
        print(f'Using {_name} DML notebook columns: {len(_cols)} from {_path}')
    else:
        if _name == 'HYPERION':
            print(f'Using {_name} all numeric effective predictor columns; no explicit DML columns list found')
        else:
            print(f'Using {_name} fallback columns: {len(CURATED[_name])}; no explicit DML columns list found')


Using eICU DML notebook columns: 41 from /home/mbranda1/ttmhte/eICU/eICUAnalysisDML.ipynb
Using PMAP DML notebook columns: 341 from /home/mbranda1/ttmhte/pmap/PMAPAnalysisDML.ipynb
Using MIMIC-IV DML notebook columns: 346 from /home/mbranda1/ttmhte/mimiciv/MIMICAnalysisDML.ipynb
Using HYPERION all numeric effective predictor columns; no explicit DML columns list found


## Dataset loaders

Loaders replicate the current cohort filters and canonicalize treatment/outcome columns to:
`TTM`, `mortality`, and `neuro_favorable`.


In [25]:
UNSCORABLE = 'Unable to score due to medication'


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Missing {path}. Restore the predictor CSVs before executing the notebook.'
        )


def load_eicu():
    require_file(CSV['eICU'])
    df = pd.read_csv(CSV['eICU'])
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        if c in df.columns:
            df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df['gender'] = (df['gender'] == 'Male').astype(int)
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['nurse_first_Motor'] != 6) & ~df['Hypothermia'].isna()].copy()
    return df.rename(columns={
        'Hypothermia': 'TTM',
        'DeathAtDischarge': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def _load_epic_style(name):
    require_file(CSV[name])
    df = pd.read_csv(CSV[name])
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    df = df[(df['first_mGCS'] != 6) & ~df['hypothermia'].isna()].copy()
    return df.rename(columns={
        'hypothermia': 'TTM',
        'death_at_disch': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def load_pmap():
    return _load_epic_style('PMAP')


def load_mimic():
    return _load_epic_style('MIMIC-IV')


def load_hyperion():
    require_file(CSV['HYPERION'])
    df = pd.read_csv(CSV['HYPERION'])
    df = df[df['groupe'] != 2].copy()
    df['TTM'] = (df['groupe'] == 1).astype(int)
    return df.rename(columns={
        'hospital_mortality': 'mortality',
        'CPC12': 'neuro_favorable',
    })


LOADERS = {
    'eICU': load_eicu,
    'PMAP': load_pmap,
    'MIMIC-IV': load_mimic,
    'HYPERION': load_hyperion,
}


def effective_feature_source(df, dataset):
    drop = {'TTM', 'mortality', 'neuro_favorable'}
    if dataset == 'eICU':
        temp_cols = [c for c in df.columns if 'emp' in c]
        drop.update(temp_cols)
        drop.update([
            'FirstGCS', 'LastMGCSTime', 'FirstMGCSTime', 'LastMGCSPositive', 'LastMGCS',
            'apacheadmissiondx', 'hospitaladmittime24', 'FirstGCSTime', 'LastGCSTime',
            'LastGCS', 'hospitaldischargestatus', 'LastGCS15', 'hospitaladmitsource',
            'DeathAtDischarge', 'Hypothermia', 'patientunitstayid', 'hypothermia_time',
        ])
    elif dataset == 'PMAP':
        temp_cols = [c for c in df.columns if ('emp' in c and c[:3] == 'dx_')]
        drop.update(temp_cols)
        drop.update([
            'first_mGCS', 'last_mGCS_time', 'first_mGCS_time', 'LastMGCSPositive',
            'last_mGCS', 'death_at_disch', 'hypothermia',
        ])
    elif dataset == 'MIMIC-IV':
        temp_cols = [c for c in df.columns if ('emp' in c or c[:3] == 'dx_')]
        drop.update(temp_cols)
        drop.update([
            'last_mGCS_time', 'first_mGCS_time', 'LastMGCSPositive',
            'last_mGCS', 'death_at_disch', 'hypothermia',
        ])
    elif dataset == 'HYPERION':
        # HYPERION: use all available numeric predictor columns. Exclude only treatment,
        # identifiers, endpoints, and clearly post-outcome/follow-up variables.
        drop.update([
            'CPC_SC3', 'CPC12', 'SUBJID', 'BARTHEL_SC', 'SOFA_SC7', 'DS_DC',
            'DAYS_ALIVE_30', 'hospital_mortality', 'groupe',
        ])

    X = df.drop(columns=[c for c in drop if c in df.columns], errors='ignore')
    X = X.select_dtypes(exclude=['object'])
    if dataset != 'HYPERION':
        binary_cols = X.columns[X.nunique(dropna=True) == 2]
        low_cols = X[binary_cols].sum(numeric_only=True)
        low_cols = low_cols[low_cols < 15].index.tolist()
        X = X.drop(columns=low_cols, errors='ignore')
    return X


def load_analysis_frame(dataset, outcome):
    df = LOADERS[dataset]()
    feature_source = effective_feature_source(df, dataset)
    if dataset in PARSED_DML_COLUMN_DATASETS:
        cols = sorted(set(c for c in CURATED[dataset] if c in feature_source.columns))
    elif dataset == 'HYPERION':
        cols = list(feature_source.columns)
    else:
        cols = list(feature_source.columns)
    if dataset in PARSED_DML_COLUMN_DATASETS:
        missing = [c for c in CURATED[dataset] if c not in df.columns]
        excluded_by_dml = [c for c in CURATED[dataset] if c in df.columns and c not in feature_source.columns]
    else:
        missing = []
        excluded_by_dml = []
    X = feature_source[cols].apply(pd.to_numeric, errors='coerce')
    T = pd.to_numeric(df['TTM'], errors='coerce')
    y = pd.to_numeric(df[outcome], errors='coerce')
    keep = T.notna() & y.notna()
    out = X.loc[keep].reset_index(drop=True)
    out['TTM'] = T.loc[keep].astype(int).reset_index(drop=True)
    out[outcome] = y.loc[keep].astype(int).reset_index(drop=True)
    meta = {
        'dataset': dataset,
        'outcome': outcome,
        'n': int(keep.sum()),
        'n_ttm': int(T.loc[keep].sum()),
        'n_features': len(cols),
        'missing_curated_features': missing,
        'dml_columns_excluded_by_loader': excluded_by_dml,
    }
    return out, cols, meta


cohort_rows = []
for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        try:
            _, _, meta = load_analysis_frame(ds['name'], outcome_col)
            meta['outcome_name'] = outcome_name
            cohort_rows.append(meta)
        except Exception as e:
            cohort_rows.append({
                'dataset': ds['name'],
                'outcome': outcome_col,
                'outcome_name': outcome_name,
                'status': f'missing: {e}',
            })

cohort_table = pd.DataFrame(cohort_rows)
cohort_table.to_csv(os.path.join(OUTDIR, 'cohort_table.csv'), index=False)
display(cohort_table)


,dataset,outcome,n,n_ttm,n_features,missing_curated_features,dml_columns_excluded_by_loader,outcome_name
0,eICU,mortality,1842,608,36,[],"[patientunitstayid, nurse_first_Temperature (C...",mortality
1,eICU,neuro_favorable,1842,608,36,[],"[patientunitstayid, nurse_first_Temperature (C...",neuro
2,PMAP,mortality,1412,666,217,"[death_at_disch, death_at_disch, death_at_disc...","[flo_r_ed_ecg_proc_md_name_Continuously, med_o...",mortality
3,PMAP,neuro_favorable,1289,619,217,"[death_at_disch, death_at_disch, death_at_disc...","[flo_r_ed_ecg_proc_md_name_Continuously, med_o...",neuro
4,MIMIC-IV,mortality,611,280,180,"[death_at_disch, death_at_disch, hypothermia, ...","[chart_first_total_protein, output_first_total...",mortality
5,MIMIC-IV,neuro_favorable,560,280,180,"[death_at_disch, death_at_disch, hypothermia, ...","[chart_first_total_protein, output_first_total...",neuro
6,HYPERION,mortality,581,284,159,[],[],mortality
7,HYPERION,neuro_favorable,581,284,159,[],[],neuro


## Preprocessing and common metrics

The key reviewer-facing point: all learned preprocessing is fit on training data only.


In [26]:
def make_stratify(y, T):
    return pd.Series(y).astype(str) + '_' + pd.Series(T).astype(str)


def fit_preprocessor(X_train):
    numeric_cols = [c for c in X_train.columns if X_train[c].dropna().nunique() > 2]
    binary_cols = [c for c in X_train.columns if c not in numeric_cols]
    prep = {
        'numeric_cols': numeric_cols,
        'binary_cols': binary_cols,
        'output_cols': numeric_cols + binary_cols,
        'num_scaler': None,
        'num_imputer': None,
        'bin_imputer': None,
    }
    if numeric_cols:
        prep['num_scaler'] = StandardScaler().fit(X_train[numeric_cols])
        scaled = pd.DataFrame(
            prep['num_scaler'].transform(X_train[numeric_cols]),
            columns=numeric_cols,
            index=X_train.index,
        )
        prep['num_imputer'] = KNNImputer(n_neighbors=10, keep_empty_features=True).fit(scaled)
    if binary_cols:
        prep['bin_imputer'] = KNNImputer(n_neighbors=10, keep_empty_features=True).fit(X_train[binary_cols])
    return prep


def transform_preprocessor(prep, X):
    parts = []
    if prep['numeric_cols']:
        Xn = pd.DataFrame(
            prep['num_scaler'].transform(X[prep['numeric_cols']]),
            columns=prep['numeric_cols'],
            index=X.index,
        )
        Xn = pd.DataFrame(
            prep['num_imputer'].transform(Xn),
            columns=prep['numeric_cols'],
            index=X.index,
        )
        parts.append(Xn)
    if prep['binary_cols']:
        Xb = pd.DataFrame(
            prep['bin_imputer'].transform(X[prep['binary_cols']]),
            columns=prep['binary_cols'],
            index=X.index,
        )
        parts.append(Xb)
    if not parts:
        return pd.DataFrame(index=X.index)
    return pd.concat(parts, axis=1)[prep['output_cols']].reset_index(drop=True)


def transform_fold(df, feature_cols, train_idx, val_idx, outcome_col):
    X_train_raw = df.loc[train_idx, feature_cols]
    X_val_raw = df.loc[val_idx, feature_cols]
    prep = fit_preprocessor(X_train_raw)
    X_train = transform_preprocessor(prep, X_train_raw)
    X_val = transform_preprocessor(prep, X_val_raw)
    y_train = df.loc[train_idx, outcome_col].astype(int).to_numpy()
    y_val = df.loc[val_idx, outcome_col].astype(int).to_numpy()
    T_train = df.loc[train_idx, 'TTM'].astype(int).to_numpy()
    T_val = df.loc[val_idx, 'TTM'].astype(int).to_numpy()
    return X_train, X_val, y_train, y_val, T_train, T_val, prep


def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    y = np.asarray(y)
    p = np.asarray(p)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    if not aucs:
        return np.nan, np.nan
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def calibration_slope(y, p):
    p = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
    logit_p = np.log(p / (1 - p))
    X = sm.add_constant(logit_p)
    try:
        m = sm.Logit(y, X).fit(disp=False)
        return float(m.params[1])
    except Exception:
        return np.nan


def lrt_cate_interaction(y, T, cate):
    cate = np.asarray(cate, float)
    if np.ptp(cate) < 1e-12:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': 'CATE constant'}
    d = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float), 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    y = np.asarray(y, float)
    try:
        m0 = sm.Logit(y, d[['const', 'T', 'cate']]).fit(disp=False)
        m1 = sm.Logit(y, d[['const', 'T', 'cate', 'tx']]).fit(disp=False)
        lr = 2 * (m1.llf - m0.llf)
        return {'lr_stat': float(lr), 'p': float(chi2.sf(lr, 1)), 'note': ''}
    except Exception as e:
        return {'lr_stat': np.nan, 'p': np.nan, 'note': str(e)}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    T = np.asarray(T, float)
    y = np.asarray(y, float)
    cate = np.asarray(cate, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    if np.ptp(cate) < 1e-12:
        return {'wald_p': np.nan, 'note': 'CATE constant'}
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    d = pd.DataFrame({'const': 1.0, 'T': T, 'cate': cate})
    d['tx'] = d['T'] * d['cate']
    try:
        m = sm.GLM(y, d[['const', 'T', 'cate', 'tx']], family=sm.families.Binomial(),
                   freq_weights=w).fit(cov_type='HC1')
        return {'wald_p': float(m.pvalues['tx']), 'note': ''}
    except Exception as e:
        return {'wald_p': np.nan, 'note': str(e)}


def run_gates(cate, y, T, ps=None, observational=False, title='', save_prefix=None):
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float), 'cate': np.asarray(cate, float)})
    if observational and ps is not None:
        ps = np.clip(np.asarray(ps, float), *PS_CLIP)
        d['w'] = np.where(d['T'] == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    else:
        d['w'] = 1.0
    d['group'] = pd.qcut(d['cate'].rank(method='first'), q=N_GATES_GROUPS, labels=False) + 1
    rows = []
    for g, sub in d.groupby('group'):
        m = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = np.asarray(m.conf_int())
        rows.append({
            'group': int(g),
            'n': len(sub),
            'mean_cate': sub['cate'].mean(),
            'gate': m.params.iloc[1],
            'ci_low': ci[1, 0],
            'ci_high': ci[1, 1],
        })
    gdf = pd.DataFrame(rows)
    gd = pd.get_dummies(d['group'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])
    if save_prefix:
        fig, ax = plt.subplots(figsize=(5.8, 4.0))
        ax.errorbar(gdf['group'], gdf['gate'],
                    yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                    fmt='o-', capsize=4)
        ax.axhline(0, color='0.4', ls='--', lw=1)
        ax.set_xlabel('Predicted CATE quintile')
        ax.set_ylabel('Group average treatment effect')
        ax.set_title(f'{title}\nF p={f_p:.3f}; rho={rho:.2f} (p={rho_p:.3f})')
        fig.tight_layout()
        fig.savefig(os.path.join(OUTDIR, f'{save_prefix}_gates.png'), dpi=180)
        plt.close(fig)
    return gdf, {'f_stat': float(f_stat), 'f_p': float(f_p), 'spearman_rho': float(rho), 'spearman_p': float(rho_p)}


## Model definitions

In [27]:
def slearner_design(X, T):
    Xd = X.copy()
    Xd['TTM'] = np.asarray(T, float)
    return Xd


def predict_slearner_cate(model, X):
    x0 = X.copy()
    x1 = X.copy()
    x0['TTM'] = 0.0
    x1['TTM'] = 1.0
    p0 = model.predict_proba(x0)[:, 1]
    p1 = model.predict_proba(x1)[:, 1]
    return p1 - p0, p0, p1


def fit_xgb_slearner(X_train, T_train, y_train):
    model = XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1)
    grid = {
        'n_estimators': [25, 50, 100, 200],
        'max_depth': [2, 5, 10],
        'learning_rate': [0.03, 0.1],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    gs = GridSearchCV(model, grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
    gs.fit(slearner_design(X_train, T_train), y_train)
    return gs.best_estimator_, gs.best_params_


def fit_lasso_slearner(X_train, T_train, y_train):
    Xd = slearner_design(X_train, T_train)
    base_cols = list(X_train.columns)
    for c in base_cols:
        Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
    model = LogisticRegressionCV(
        penalty='l1',
        solver='saga',
        Cs=[1e-3, 1e-2, 1e-1, 1.0],
        cv=5,
        scoring='neg_log_loss',
        max_iter=20000,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(Xd, y_train)
    return model, {'chosen_C': float(model.C_[0]), 'interaction_cols': [f'{c}_x_TTM' for c in base_cols]}


def predict_lasso_slearner(model, X, interaction_cols):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        for c in X.columns:
            Xd[f'{c}_x_TTM'] = Xd[c] * Xd['TTM']
        return Xd
    p0 = model.predict_proba(build(0))[:, 1]
    p1 = model.predict_proba(build(1))[:, 1]
    return p1 - p0, p0, p1


def fit_neural_slearner(X_train, T_train, y_train, dataset):
    nn_cfg = {
        'eICU':     {'layers': [64, 64, 32],       'dropout': 0.5, 'lr': 1e-3, 'epochs': 20},
        'PMAP':     {'layers': [64, 64, 64, 64, 32],'dropout': 0.2, 'lr': 1e-4, 'epochs': 50},
        'MIMIC-IV': {'layers': [64, 64, 64, 32],   'dropout': 0.2, 'lr': 1e-3, 'epochs': 60},
        'HYPERION': {'layers': [64],               'dropout': 0.5, 'lr': 1e-3, 'epochs': 15},
    }[dataset]
    Xd_df = slearner_design(X_train, T_train)
    if NEURAL_BACKEND != 'keras' or keras is None:
        # Cluster-safe CPU neural net. This avoids TensorFlow CUDA/PTX failures while preserving
        # the S-learner neural-network sensitivity row.
        model = MLPClassifier(
            hidden_layer_sizes=tuple(nn_cfg['layers']),
            activation='relu',
            solver='adam',
            learning_rate_init=nn_cfg['lr'],
            alpha=1e-4,
            batch_size=32,
            max_iter=max(100, nn_cfg['epochs'] * 5),
            early_stopping=True,
            n_iter_no_change=10,
            random_state=SEED,
        )
        model.fit(Xd_df, np.asarray(y_train).astype(int))
        out_cfg = dict(nn_cfg)
        out_cfg['backend'] = 'sklearn_mlp'
        return model, out_cfg

    tf.keras.utils.set_random_seed(SEED)
    Xd = Xd_df.to_numpy(dtype='float32')
    y = np.asarray(y_train).astype('float32')
    inp = keras.Input(shape=(Xd.shape[1],))
    z = inp
    for width in nn_cfg['layers']:
        z = layers.Dense(width, activation='relu')(z)
        z = layers.Dropout(nn_cfg['dropout'])(z)
    out = layers.Dense(1, activation='sigmoid')(z)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=nn_cfg['lr']),
                  loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')],
                  jit_compile=False)
    if USE_TF_GPU:
        model.fit(Xd, y, batch_size=32, epochs=nn_cfg['epochs'], validation_split=0.2, verbose=0)
    else:
        with tf.device('/CPU:0'):
            model.fit(Xd, y, batch_size=32, epochs=nn_cfg['epochs'], validation_split=0.2, verbose=0)
    out_cfg = dict(nn_cfg)
    out_cfg['backend'] = 'keras'
    return model, out_cfg


def predict_neural_slearner(model, X):
    def build_df(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd
    if hasattr(model, 'predict_proba'):
        p0 = model.predict_proba(build_df(0))[:, 1]
        p1 = model.predict_proba(build_df(1))[:, 1]
    else:
        p0 = model.predict(build_df(0).to_numpy(dtype='float32'), verbose=0).ravel()
        p1 = model.predict(build_df(1).to_numpy(dtype='float32'), verbose=0).ravel()
    return p1 - p0, p0, p1


def fit_bart_slearner(X_train, T_train, y_train, dataset):
    if pm is None or pmb is None:
        raise RuntimeError('PyMC/pymc_bart is not available')
    Xd = slearner_design(X_train, T_train).to_numpy(dtype='float64')
    y = np.asarray(y_train).astype(int)
    m_trees = {'eICU': 30, 'PMAP': 30, 'MIMIC-IV': 50, 'HYPERION': 30}[dataset]
    with pm.Model() as model:
        X_shared = pm.Data('X_shared', Xd)
        u = pmb.BART('u', X=X_shared, Y=y, m=m_trees)
        p = pm.Deterministic('p', pm.math.sigmoid(u))
        pm.Bernoulli('y', p=p, observed=y)
        trace = pm.sample(
            draws=BART_DRAWS[dataset],
            tune=BART_TUNE,
            chains=BART_CHAINS,
            cores=BART_CORES,
            random_seed=SEED,
            progressbar=True,
        )
    return model, trace, {'m_trees': m_trees, 'draws': BART_DRAWS[dataset], 'tune': BART_TUNE}


def predict_bart_slearner(model, trace, X):
    def build(T_value):
        Xd = X.copy()
        Xd['TTM'] = float(T_value)
        return Xd.to_numpy(dtype='float64')
    preds = []
    with model:
        for val in [0, 1]:
            pm.set_data({'X_shared': build(val)})
            ppc = pm.sample_posterior_predictive(trace, var_names=['p'], random_seed=SEED, progressbar=False)
            arr = np.asarray(ppc.posterior_predictive['p'])
            preds.append(arr.reshape(-1, arr.shape[-1]).mean(axis=0))
    p0, p1 = preds
    return p1 - p0, p0, p1


def fit_causal_forest(X_train, T_train, y_train, dataset):
    if CausalForestDML is None:
        raise RuntimeError('econml is not available')
    # Keep HYPERION settings aligned with the submitted DML notebook; observational settings
    # match the current revision sensitivity notebooks.
    if dataset == 'HYPERION':
        model_y = XGBClassifier(max_depth=25, n_estimators=100, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=10, n_estimators=20, random_state=SEED, eval_metric='logloss')
    elif dataset == 'PMAP':
        model_y = XGBClassifier(max_depth=5, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    else:
        model_y = XGBClassifier(max_depth=3, n_estimators=50, random_state=SEED, eval_metric='logloss')
        model_t = XGBClassifier(max_depth=2, n_estimators=20, random_state=SEED, eval_metric='logloss')
    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        discrete_outcome=True,
        random_state=SEED,
        n_jobs=-1,
    )
    cf.fit(y_train, T_train, X=X_train, cache_values=True)
    return cf, {'model_y': str(model_y), 'model_t': str(model_t)}


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


## Run all datasets, outcomes, and models

In [28]:
RESULTS = []
GATES_ROWS = []
IMPORTANCE_ROWS = []


def add_result(row):
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(os.path.join(OUTDIR, 'all_model_results_partial.csv'), index=False)


def evaluate_oof_slearner(dataset, outcome_name, model_name, cate, p_obs, y, T, extra=None):
    lrt = lrt_cate_interaction(y, T, cate)
    auc = roc_auc_score(y, p_obs) if len(np.unique(y)) == 2 else np.nan
    ci_low, ci_high = bootstrap_auc_ci(y, p_obs)
    row = {
        'dataset': dataset,
        'outcome': outcome_name,
        'model': model_name,
        'n_eval': len(y),
        'cv_splits': N_SPLITS,
        'lr_p': lrt['p'],
        'lr_note': lrt['note'],
        'auroc': auc,
        'auroc_ci_low': ci_low,
        'auroc_ci_high': ci_high,
        'brier': brier_score_loss(y, p_obs),
        'calibration_slope': calibration_slope(y, p_obs),
        'mean_cate': float(np.mean(cate)),
        'cate_p05': float(np.percentile(cate, 5)),
        'cate_p50': float(np.percentile(cate, 50)),
        'cate_p95': float(np.percentile(cate, 95)),
    }
    if extra:
        row.update(extra)
    add_result(row)


def run_dataset_outcome(dataset, outcome_name, outcome_col, observational):
    print(f'\n===== {dataset} / {outcome_name} =====')
    try:
        df, feature_cols, meta = load_analysis_frame(dataset, outcome_col)
    except Exception as e:
        add_result({
            'dataset': dataset,
            'outcome': outcome_name,
            'model': 'all',
            'status': f'skipped before modeling: {e}',
        })
        print('Skipped:', e)
        return
    y_all = df[outcome_col].astype(int).to_numpy()
    T_all = df['TTM'].astype(int).to_numpy()
    n = len(df)
    print(f'n={n}; {N_SPLITS}-fold stratified CV; features={len(feature_cols)}')

    oof = {}
    for model_name in ['S-learner XGBoost', 'S-learner L1 logistic', 'S-learner neural network',
                       'S-learner BART', 'CausalForestDML/R-learner']:
        oof[model_name] = {
            'cate': np.full(n, np.nan),
            'p_obs': np.full(n, np.nan),
            'lo': np.full(n, np.nan),
            'hi': np.full(n, np.nan),
            'ps': np.full(n, np.nan),
            'extras': [],
        }

    strat = make_stratify(y_all, T_all)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(n), strat), start=1):
        print(f'  fold {fold}/{N_SPLITS}: train={len(train_idx)}, validation={len(val_idx)}')
        X_train, X_val, y_train, y_val, T_train, T_val, prep = transform_fold(
            df, feature_cols, train_idx, val_idx, outcome_col
        )

        if RUN_XGB_SLEARNER:
            try:
                model, params = fit_xgb_slearner(X_train, T_train, y_train)
                cate, p0, p1 = predict_slearner_cate(model, X_val)
                oof['S-learner XGBoost']['cate'][val_idx] = cate
                oof['S-learner XGBoost']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner XGBoost']['extras'].append(params)
                if hasattr(model, 'feature_importances_'):
                    for col, val in zip(slearner_design(X_train, T_train).columns, model.feature_importances_):
                        IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name, 'fold': fold,
                                                'model': 'S-learner XGBoost', 'feature': col, 'importance': float(val)})
            except Exception as e:
                print(f'    XGBoost failed fold {fold}:', e)

        if RUN_LASSO_SLEARNER:
            try:
                model, info = fit_lasso_slearner(X_train, T_train, y_train)
                cate, p0, p1 = predict_lasso_slearner(model, X_val, info['interaction_cols'])
                oof['S-learner L1 logistic']['cate'][val_idx] = cate
                oof['S-learner L1 logistic']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                coefs = pd.Series(model.coef_[0], index=slearner_design(X_train, T_train).columns.tolist() + info['interaction_cols'])
                oof['S-learner L1 logistic']['extras'].append({
                    'chosen_C': info['chosen_C'],
                    'nonzero_interactions': int((coefs[info['interaction_cols']] != 0).sum()),
                    'n_interactions': len(info['interaction_cols']),
                })
            except Exception as e:
                print(f'    L1 logistic failed fold {fold}:', e)

        if RUN_NEURAL_SLEARNER:
            try:
                model, cfg = fit_neural_slearner(X_train, T_train, y_train, dataset)
                cate, p0, p1 = predict_neural_slearner(model, X_val)
                oof['S-learner neural network']['cate'][val_idx] = cate
                oof['S-learner neural network']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner neural network']['extras'].append(cfg)
            except Exception as e:
                print(f'    Neural failed fold {fold}:', e)

        if RUN_BART_SLEARNER:
            try:
                model, trace, cfg = fit_bart_slearner(X_train, T_train, y_train, dataset)
                cate, p0, p1 = predict_bart_slearner(model, trace, X_val)
                oof['S-learner BART']['cate'][val_idx] = cate
                oof['S-learner BART']['p_obs'][val_idx] = np.where(T_val == 1, p1, p0)
                oof['S-learner BART']['extras'].append(cfg)
            except Exception as e:
                print(f'    BART failed fold {fold}:', e)

        if RUN_CAUSAL_FOREST:
            try:
                cf, cfg = fit_causal_forest(X_train, T_train, y_train, dataset)
                cate = np.ravel(cf.effect(X_val))
                lo, hi = cf.effect_interval(X_val, alpha=0.05)
                oof['CausalForestDML/R-learner']['cate'][val_idx] = cate
                oof['CausalForestDML/R-learner']['lo'][val_idx] = np.ravel(lo)
                oof['CausalForestDML/R-learner']['hi'][val_idx] = np.ravel(hi)
                oof['CausalForestDML/R-learner']['ps'][val_idx] = (
                    cf_propensity(cf, X_val) if observational else np.repeat(T_train.mean(), len(val_idx))
                )
                oof['CausalForestDML/R-learner']['extras'].append(cfg)
                if hasattr(cf, 'feature_importances_'):
                    for col, val in zip(X_train.columns, cf.feature_importances_):
                        IMPORTANCE_ROWS.append({'dataset': dataset, 'outcome': outcome_name, 'fold': fold,
                                                'model': 'CausalForestDML/R-learner',
                                                'feature': col, 'importance': float(val)})
            except Exception as e:
                print(f'    CausalForest failed fold {fold}:', e)

    for model_name, d in oof.items():
        ran = ~np.isnan(d['cate'])
        if not ran.any():
            add_result({'dataset': dataset, 'outcome': outcome_name, 'model': model_name,
                        'status': 'failed: no folds produced predictions'})
            continue
        if model_name == 'CausalForestDML/R-learner':
            lrt = lrt_cate_interaction(y_all[ran], T_all[ran], d['cate'][ran])
            ps = d['ps'][ran]
            ipw = ipw_interaction_test(y_all[ran], T_all[ran], d['cate'][ran], ps) if observational else {'wald_p': np.nan, 'note': 'randomized'}
            gates, gates_stats = run_gates(
                d['cate'][ran], y_all[ran], T_all[ran], ps=ps, observational=observational,
                title=f'{dataset} {outcome_name} CausalForestDML ({N_SPLITS}-fold OOF)',
                save_prefix=f'{dataset}_{outcome_name}_causal_forest_cv'.replace(' ', '_').replace('/', '_')
            )
            gates.insert(0, 'dataset', dataset)
            gates.insert(1, 'outcome', outcome_name)
            gates.insert(2, 'model', model_name)
            GATES_ROWS.extend(gates.to_dict('records'))
            row = {
                'dataset': dataset,
                'outcome': outcome_name,
                'model': model_name,
                'n_eval': int(ran.sum()),
                'cv_splits': N_SPLITS,
                'lr_p': lrt['p'],
                'lr_note': lrt['note'],
                'ipw_interaction_p': ipw['wald_p'],
                'gates_f_p': gates_stats['f_p'],
                'gates_spearman_rho': gates_stats['spearman_rho'],
                'gates_spearman_p': gates_stats['spearman_p'],
                'pct_cate_ci_cross_zero': float(np.mean((d['lo'][ran] < 0) & (d['hi'][ran] > 0))),
                'mean_cate': float(np.mean(d['cate'][ran])),
                'cate_p05': float(np.percentile(d['cate'][ran], 5)),
                'cate_p50': float(np.percentile(d['cate'][ran], 50)),
                'cate_p95': float(np.percentile(d['cate'][ran], 95)),
            }
            add_result(row)
        else:
            extra = {}
            if model_name == 'S-learner L1 logistic' and d['extras']:
                extra = {
                    'chosen_C': float(np.median([x['chosen_C'] for x in d['extras']])),
                    'nonzero_interactions': float(np.median([x['nonzero_interactions'] for x in d['extras']])),
                    'n_interactions': int(np.median([x['n_interactions'] for x in d['extras']])),
                }
            evaluate_oof_slearner(dataset, outcome_name, model_name,
                                  d['cate'][ran], d['p_obs'][ran], y_all[ran], T_all[ran], extra)


for ds in DATASETS:
    for outcome_name, outcome_col in OUTCOMES.items():
        run_dataset_outcome(ds['name'], outcome_name, outcome_col, ds['observational'])

results = pd.DataFrame(RESULTS)
gates_table = pd.DataFrame(GATES_ROWS)
importance_table = pd.DataFrame(IMPORTANCE_ROWS)

results.to_csv(os.path.join(OUTDIR, 'all_model_results.csv'), index=False)
gates_table.to_csv(os.path.join(OUTDIR, 'gates_results.csv'), index=False)
importance_table.to_csv(os.path.join(OUTDIR, 'variable_importance.csv'), index=False)

display(results)



===== eICU / mortality =====
n=1842; 5-fold stratified CV; features=36
  fold 1/5: train=1473, validation=369


Only 50 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Sequential sampling (2 chains in 1 job)
PGBART: [u]


Sampling 2 chains for 50 tune and 50 draw iterations (100 + 100 draws total) took 5 seconds.
The number of samples is too small to check convergence reliably.
Sampling: [u]
Sampling: [u]


  fold 2/5: train=1473, validation=369


Only 50 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.
Sequential sampling (2 chains in 1 job)
PGBART: [u]


...

Sampling 2 chains for 50 tune and 100 draw iterations (100 + 200 draws total) took 4 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [u]
Sampling: [u]


  fold 5/5: train=465, validation=116


Sequential sampling (2 chains in 1 job)
PGBART: [u]


Sampling 2 chains for 50 tune and 100 draw iterations (100 + 200 draws total) took 4 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [u]
Sampling: [u]


,dataset,outcome,model,n_eval,cv_splits,lr_p,lr_note,auroc,auroc_ci_low,auroc_ci_high,brier,calibration_slope,mean_cate,cate_p05,cate_p50,cate_p95,chosen_C,nonzero_interactions,n_interactions,ipw_interaction_p,gates_f_p,gates_spearman_rho,gates_spearman_p,pct_cate_ci_cross_zero
0,eICU,mortality,S-learner XGBoost,1842,5,0.247154,,0.727743,0.706525,0.751141,0.209557,1.091684,0.008248,0.000000,6.376117e-03,2.760507e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,eICU,mortality,S-learner L1 logistic,1842,5,0.000171,,0.736214,0.715510,0.758404,0.206587,1.012332,0.021226,-0.076425,1.963634e-02,1.282753e-01,0.1,9.0,36.0,NaN,NaN,NaN,NaN,NaN
2,eICU,mortality,S-learner neural network,1842,5,0.167298,,0.714719,0.692074,0.736853,0.216455,0.675413,0.039973,-0.008985,1.977470e-02,1.705739e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,eICU,mortality,S-learner BART,1842,5,0.508038,,0.731511,0.709487,0.754138,0.209041,1.201186,0.007864,-0.008240,4.830104e-03,3.771765e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,eICU,mortality,CausalForestDML/R-learner,1842,5,0.034867,,NaN,NaN,NaN,NaN,NaN,0.062362,-0.028908,6.203450e-02,1.572744e-01,NaN,NaN,NaN,0.094646,0.216548,0.6,0.284757,0.977199
5,eICU,neuro,S-learner XGBoost,1842,5,0.125476,,0.733239,0.710189,0.756848,0.192295,0.774652,-0.039126,-0.118443,-2.999015e-02,5.099048e-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,eICU,neuro,S-learner L1 logistic,1842,5,0.003561,,0.735774,0.714224,0.758630,0.190038,0.976667,-0.029769,-0.125670,-2.775376e-02,6.494166e-02,0.1,6.0,36.0,NaN,NaN,NaN,NaN,NaN
7,eICU,neuro,S-learner neural network,1842,5,0.379768,,0.714903,0.690393,0.739582,0.199561,0.630253,-0.030234,-0.148155,-1.136047e-02,1.642943e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,eICU,neuro,S-learner BART,1842,5,0.098404,,0.747544,0.724776,0.768171,0.186615,1.198804,-0.027676,-0.050128,-2.671431e-02,-9.831424e-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,eICU,neuro,CausalForestDML/R-learner,1842,5,0.005345,,NaN,NaN,NaN,NaN,NaN,-0.071910,-0.179149,-6.895669e-02,2.472588e-02,NaN,NaN,NaN,0.015339,0.136745,0.9,0.037386,0.935939


## Paper-facing summary tables

In [29]:
results = pd.DataFrame(RESULTS)

for col in ['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier',
            'calibration_slope', 'nonzero_interactions', 'n_interactions', 'chosen_C']:
    if col not in results.columns:
        results[col] = np.nan

table2 = results.pivot_table(
    index=['dataset', 'outcome', 'model'],
    values=['lr_p', 'auroc', 'auroc_ci_low', 'auroc_ci_high', 'brier', 'calibration_slope'],
    aggfunc='first',
).reset_index()
table2.to_csv(os.path.join(OUTDIR, 'table2_model_performance_and_lr.csv'), index=False)
display(table2.round(4))

cf_summary = results[results['model'].eq('CausalForestDML/R-learner')].copy()
cf_summary.to_csv(os.path.join(OUTDIR, 'causal_forest_hte_summary.csv'), index=False)
display(cf_summary.round(4))

lasso_summary = results[results['model'].eq('S-learner L1 logistic')][
    ['dataset', 'outcome', 'nonzero_interactions', 'n_interactions', 'chosen_C', 'lr_p']
].copy()
lasso_summary.to_csv(os.path.join(OUTDIR, 'lasso_slearner_summary.csv'), index=False)
display(lasso_summary)


,dataset,outcome,model,auroc,auroc_ci_high,auroc_ci_low,brier,calibration_slope,lr_p
0,HYPERION,mortality,CausalForestDML/R-learner,NaN,NaN,NaN,NaN,NaN,0.9454
1,HYPERION,mortality,S-learner BART,0.7931,0.8354,0.7456,0.1244,1.3451,0.4796
2,HYPERION,mortality,S-learner L1 logistic,0.7893,0.8369,0.7408,0.1243,1.2362,0.1863
3,HYPERION,mortality,S-learner XGBoost,0.7725,0.8184,0.7216,0.1266,0.7893,NaN
4,HYPERION,mortality,S-learner neural network,0.7075,0.7573,0.6571,0.1427,0.5064,0.0343
5,HYPERION,neuro,CausalForestDML/R-learner,NaN,NaN,NaN,NaN,NaN,0.3422
6,HYPERION,neuro,S-learner BART,0.8536,0.8996,0.8002,0.0610,1.7140,0.6582
7,HYPERION,neuro,S-learner L1 logistic,0.8146,0.8742,0.7461,0.0636,1.9952,0.3026
8,HYPERION,neuro,S-learner XGBoost,0.7949,0.8552,0.7245,0.0620,0.6443,0.0250
9,HYPERION,neuro,S-learner neural network,0.6068,0.6914,0.5228,0.0747,0.4050,0.1261


,dataset,outcome,model,n_eval,cv_splits,lr_p,lr_note,auroc,auroc_ci_low,auroc_ci_high,brier,calibration_slope,mean_cate,cate_p05,cate_p50,cate_p95,chosen_C,nonzero_interactions,n_interactions,ipw_interaction_p,gates_f_p,gates_spearman_rho,gates_spearman_p,pct_cate_ci_cross_zero
4,eICU,mortality,CausalForestDML/R-learner,1842,5,0.0349,,NaN,NaN,NaN,NaN,NaN,0.0624,-0.0289,0.0620,0.1573,NaN,NaN,NaN,0.0946,0.2165,0.6,0.2848,0.9772
9,eICU,neuro,CausalForestDML/R-learner,1842,5,0.0053,,NaN,NaN,NaN,NaN,NaN,-0.0719,-0.1791,-0.0690,0.0247,NaN,NaN,NaN,0.0153,0.1367,0.9,0.0374,0.9359
14,PMAP,mortality,CausalForestDML/R-learner,1412,5,0.2001,,NaN,NaN,NaN,NaN,NaN,0.0712,-0.0045,0.0723,0.1445,NaN,NaN,NaN,0.4128,0.8116,0.2,0.7471,0.9724
19,PMAP,neuro,CausalForestDML/R-learner,1289,5,0.3828,,NaN,NaN,NaN,NaN,NaN,-0.0625,-0.1335,-0.0620,0.0069,NaN,NaN,NaN,0.1671,0.0116,-0.5,0.3910,0.9651
24,MIMIC-IV,mortality,CausalForestDML/R-learner,611,5,0.3433,,NaN,NaN,NaN,NaN,NaN,-0.0683,-0.1644,-0.0677,0.0370,NaN,NaN,NaN,0.0762,0.1079,0.6,0.2848,0.9656
29,MIMIC-IV,neuro,CausalForestDML/R-learner,560,5,0.8436,,NaN,NaN,NaN,NaN,NaN,0.0358,-0.0487,0.0362,0.1181,NaN,NaN,NaN,0.3049,0.6520,-0.6,0.2848,0.9911
34,HYPERION,mortality,CausalForestDML/R-learner,581,5,0.9454,,NaN,NaN,NaN,NaN,NaN,-0.0079,-0.0692,-0.0091,0.0558,NaN,NaN,NaN,NaN,0.8462,0.1,0.8729,0.9931
39,HYPERION,neuro,CausalForestDML/R-learner,581,5,0.3422,,NaN,NaN,NaN,NaN,NaN,0.0200,-0.0188,0.0164,0.0715,NaN,NaN,NaN,NaN,0.9523,0.1,0.8729,0.9845


,dataset,outcome,nonzero_interactions,n_interactions,chosen_C,lr_p
1,eICU,mortality,9.0,36.0,0.1,0.000171
6,eICU,neuro,6.0,36.0,0.1,0.003561
11,PMAP,mortality,105.0,217.0,1.0,0.209049
16,PMAP,neuro,94.0,217.0,1.0,0.000015
21,MIMIC-IV,mortality,82.0,180.0,1.0,0.281441
26,MIMIC-IV,neuro,84.0,180.0,1.0,0.994434
31,HYPERION,mortality,2.0,159.0,0.1,0.186320
36,HYPERION,neuro,1.0,159.0,0.1,0.302602


## Response-letter text snippets

These lines are deliberately terse so they can be pasted into `RESPONSE_TO_REVIEWERS.md`.


In [30]:
def fmt_p(x):
    if pd.isna(x):
        return 'NA'
    return '<0.001' if x < 0.001 else f'{x:.3f}'


print('--- Train/test and preprocessing ---')
print(f'Within each dataset, patients were evaluated with {N_SPLITS}-fold stratified cross-validation, stratified jointly by treatment and outcome. Standard scaling, KNN imputation (k=10), and all model tuning were fit on each fold training partition only; metrics and HTE tests use out-of-fold predictions.')

print('\n--- CausalForestDML/R-learner HTE summary ---')
for _, r in cf_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: LR p={fmt_p(r['lr_p'])}; "
          f"GATES F p={fmt_p(r['gates_f_p'])}; "
          f"CATE 95% CIs crossing zero={r['pct_cate_ci_cross_zero']:.1%}; "
          f"IPW interaction p={fmt_p(r.get('ipw_interaction_p', np.nan))}.")

print('\n--- L1 S-learner regularization summary ---')
for _, r in lasso_summary.iterrows():
    print(f"{r['dataset']} {r['outcome']}: {int(r['nonzero_interactions'])}/{int(r['n_interactions'])} treatment interactions survived the L1 penalty; LR p={fmt_p(r['lr_p'])}.")


--- Train/test and preprocessing ---
Within each dataset, patients were evaluated with 5-fold stratified cross-validation, stratified jointly by treatment and outcome. Standard scaling, KNN imputation (k=10), and all model tuning were fit on each fold training partition only; metrics and HTE tests use out-of-fold predictions.

--- CausalForestDML/R-learner HTE summary ---
eICU mortality: LR p=0.035; GATES F p=0.217; CATE 95% CIs crossing zero=97.7%; IPW interaction p=0.095.
eICU neuro: LR p=0.005; GATES F p=0.137; CATE 95% CIs crossing zero=93.6%; IPW interaction p=0.015.
PMAP mortality: LR p=0.200; GATES F p=0.812; CATE 95% CIs crossing zero=97.2%; IPW interaction p=0.413.
PMAP neuro: LR p=0.383; GATES F p=0.012; CATE 95% CIs crossing zero=96.5%; IPW interaction p=0.167.
MIMIC-IV mortality: LR p=0.343; GATES F p=0.108; CATE 95% CIs crossing zero=96.6%; IPW interaction p=0.076.
MIMIC-IV neuro: LR p=0.844; GATES F p=0.652; CATE 95% CIs crossing zero=99.1%; IPW interaction p=0.305.
HYPER

## Reviewer response strategy

Use the notebook outputs to fill the response document, but keep the narrative conservative:

1. The primary result remains the per-dataset out-of-fold cross-validated analysis. The pooled observational analysis is a sensitivity analysis because exposure ascertainment and case mix differ.
2. For observational data, explain confounding control in layers: covariate-conditioned S-learners, orthogonalized CausalForestDML nuisance models, propensity overlap diagnostics, and IPW GATES/interaction diagnostics.
3. For dimensionality concerns, emphasize curated HYPERION-anchored covariates, rare-feature and treatment-collinearity filters, training-only feature/preprocessing steps, L1 S-learner sensitivity, and the harmonized pooled analysis.
4. Be explicit that small effects in small subgroups remain underpowered; this is a limitation, not a failure of the analysis.
